# 🎯 Hull Tactical Market Prediction - Inference Server

## ⚠️ IMPORTANT: This notebook MUST run on Kaggle platform!

**This is an INFERENCE COMPETITION notebook.**
- ❌ Cannot run locally (kaggle_evaluation module not available)
- ✅ Must upload and run on Kaggle notebook environment
- ✅ Requires Kaggle's inference competition infrastructure

## 📌 Setup Instructions

### Step 1: Create ZIP file (Already done! ✅)

```bash
bash create_kaggle_zip.sh
```

This creates `prediction_market_modules.zip` containing:
- `src/` - All Python modules (data, features, models, etc.)
- `conf/params.yaml` - Configuration file

### Step 2: Upload Dataset to Kaggle

1. **Go to**: https://www.kaggle.com/datasets
2. **Click**: "New Dataset"
3. **Upload**: `prediction_market_modules.zip`
4. **Name it**: 원하는 이름 (예: `prediction-market-modules`)
5. **Click**: "Create"

### Step 3: Create New Notebook on Kaggle

1. **Go to competition**: https://www.kaggle.com/competitions/hull-tactical-market-prediction
2. **Click**: "Code" → "New Notebook"
3. **Copy this entire notebook** into the new Kaggle notebook

### Step 4: Configure Dataset Name

**⚠️ IMPORTANT**: 첫 번째 코드 셀에서 데이터셋 이름을 수정하세요!

```python
# ========== CONFIGURATION: 데이터셋 이름 (여기만 수정하세요!) ==========
DATASET_NAME = "your-dataset-name"  # Kaggle에 업로드한 이름으로 변경
# ====================================================================
```

### Step 5: Add Dataset to Notebook

1. **In Kaggle Notebook**: Click "Add Data" (right panel)
2. **Your Datasets** → Find your uploaded dataset
3. **Click**: "Add"

### Step 6: Run Notebook

1. **Click**: "Run All"
2. **Wait**: ~15 minutes for training to complete
3. **Submit**: Click "Submit to Competition"

## 🚀 What This Notebook Does:

### Training Phase (runs once):
1. ✅ Loads and preprocesses training data
2. ✅ Engineers 600+ features → selects best ~150
3. ✅ Trains LightGBM model with cross-validation
4. ✅ Stores model in memory

### Inference Phase (for each timestep):
1. ✅ Receives new market data from Kaggle
2. ✅ Applies same preprocessing + feature engineering
3. ✅ Predicts return using trained model
4. ✅ Converts to allocation (0.0 to 2.0)
5. ✅ Returns single float within 5-minute limit

## 📊 Competition Details:

- **Prediction**: Single allocation value (0 = no investment, 1 = full market, 2 = 2x leverage)
- **Response Time**: 5 minutes per prediction
- **Startup Time**: 15 minutes for initial training
- **Evaluation**: Timestep-by-timestep (streaming)


## 1️⃣ Setup Module Paths

In [ ]:
import os
import sys
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

print("="*80)
print("SETTING UP MODULE PATHS")
print("="*80)

# ========== CONFIGURATION: 데이터셋 이름 (여기만 수정하세요!) ==========
DATASET_NAME = "prediction-market-modules"
# ====================================================================

# Add kaggle_evaluation to path (from competition data)
kaggle_eval_path = Path("/kaggle/input/hull-tactical-market-prediction")
if kaggle_eval_path.exists():
    sys.path.insert(0, str(kaggle_eval_path))
    print(f"✓ Added kaggle_evaluation path: {kaggle_eval_path}")

# Add custom modules to path - try multiple possible locations
dataset_locations = [
    Path(f"/kaggle/input/{DATASET_NAME}"),
    Path(f"/kaggle/input/{DATASET_NAME}/prediction_market_modules"),
]

module_found = False
for dataset_dir in dataset_locations:
    if dataset_dir.exists():
        sys.path.insert(0, str(dataset_dir))
        print(f"✓ Added to path: {dataset_dir}")
        
        # Check if src directory exists
        src_dir = dataset_dir / "src"
        if src_dir.exists():
            print(f"✓ Found src directory: {src_dir}")
            module_found = True
            # Save for later use
            globals()['DATASET_PATH'] = str(dataset_dir)
            break
        else:
            print(f"  (src not found in {dataset_dir})")

if not module_found:
    print(f"\n⚠️  src module not found. Available files in /kaggle/input/:")
    input_dir = Path("/kaggle/input/")
    if input_dir.exists():
        for item in input_dir.iterdir():
            print(f"  📁 {item.name}")
            if item.is_dir():
                for subitem in item.iterdir():
                    print(f"    - {subitem.name}")

print("\n✅ Path setup complete!")
print(f"Dataset name: {DATASET_NAME}")
print(f"Current sys.path:")
for p in sys.path[:5]:
    print(f"  - {p}")


## 2️⃣ Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb

# Try to import kaggle_evaluation (only available in Kaggle environment)
try:
    import kaggle_evaluation.default_inference_server
    print("✓ kaggle_evaluation imported")
    KAGGLE_ENV = True
except ImportError:
    print("⚠️  kaggle_evaluation not found - this is normal if running locally")
    print("   This notebook MUST be run on Kaggle platform for submission")
    KAGGLE_ENV = False

print("="*80)
print("IMPORTING MODULES")
print("="*80)

# Import modules with error handling
DataLoader = None
FeatureEngineering = None
ReturnPredictor = None
load_config = None
Timer = None

try:
    from src.data import DataLoader
    print("✓ DataLoader imported")
except Exception as e:
    print(f"❌ DataLoader import failed: {e}")
    raise

try:
    from src.features import FeatureEngineering
    print("✓ FeatureEngineering imported")
except Exception as e:
    print(f"❌ FeatureEngineering import failed: {e}")
    raise

try:
    from src.models import ReturnPredictor
    print("✓ ReturnPredictor imported")
except Exception as e:
    print(f"❌ ReturnPredictor import failed: {e}")
    raise

try:
    from src.utils import load_config, Timer
    print("✓ Utils imported")
except Exception as e:
    print(f"❌ Utils import failed: {e}")
    raise

print("\n✅ All imports successful!")

if not KAGGLE_ENV:
    print("\n" + "="*80)
    print("⚠️  WARNING: Not running in Kaggle environment!")
    print("="*80)
    print("This notebook requires Kaggle's inference competition environment.")
    print("Please upload and run this notebook on Kaggle platform.")
    print("="*80)


## 3️⃣ Load and Train Model (Once)

This cell trains the model and stores it in global variables.
The model will be loaded once and reused for all predictions.

In [ ]:
print("="*80)
print("FULL TRAINING PIPELINE")
print("="*80)

# Configuration - uses DATASET_PATH from first cell
config_path = f"{DATASET_PATH}/conf/params.yaml"
config = load_config(config_path)
print(f"✓ Configuration loaded from: {config_path}")

# ==================== STEP 1: Load and Preprocess Data ====================
print("\n" + "="*80)
print("STEP 1: DATA LOADING AND PREPROCESSING")
print("="*80)

train_path = "/kaggle/input/hull-tactical-market-prediction/train.csv"
data_loader = DataLoader(config_path=config_path)
train_df, _ = data_loader.load_data(train_path, train_path)
print(f"✓ Training data loaded: {train_df.shape}")

# Preprocess
train_df, _ = data_loader.preprocess_timeseries(
    train_df,
    train_df=None,
    add_missing_indicators=True,
    missing_threshold=0.1,
    add_regime_indicators=True,
    auto_detect_regime=True,
    handle_outliers=True,
    winsorize_limits=(0.001, 0.001),
    winsorize_method='rolling',
    normalize=True,
    normalize_method='rank_gauss',
    scale=True,
    scale_method='robust',
    window=60
)
print(f"✓ Preprocessing complete: {train_df.shape}")

# ==================== STEP 2: Feature Engineering ====================
print("\n" + "="*80)
print("STEP 2: FEATURE ENGINEERING")
print("="*80)

fe = FeatureEngineering(config_path=config_path)
train_features = fe.fit_transform(train_df)
print(f"✓ Features engineered: {train_features.shape}")

# ==================== STEP 3: Feature Selection ====================
print("\n" + "="*80)
print("STEP 3: FEATURE SELECTION FOR RETURN MODEL")
print("="*80)

train_selected, selected_features = fe.select_features_by_importance(
    train_features,
    target_col='forward_returns',
    method='correlation',
    top_n=200
)

# Remove correlated features
train_selected, _ = fe.remove_correlated_features(
    train_selected,
    threshold=0.95,
    target_col='forward_returns'
)

FEATURE_COLS = [
    col for col in train_selected.columns 
    if col not in ['date_id', 'forward_returns', 'risk_free_rate', 'market_forward_excess_returns']
]
print(f"✓ Feature selection complete: {len(FEATURE_COLS)} features")

# ==================== STEP 4: Train Return Model ====================
print("\n" + "="*80)
print("STEP 4: TRAINING RETURN PREDICTION MODEL")
print("="*80)

from src.models import ReturnPredictor
predictor = ReturnPredictor(model_type='lightgbm', config_path=config_path)
results = predictor.train_cv(
    df=train_selected,
    target_col='forward_returns',
    date_col='date_id'
)

print(f"\n✅ Return model trained!")
print(f"OOF Score: {results['oof_score']:.6f}")

# ==================== STEP 5: Create Risk Labels ====================
print("\n" + "="*80)
print("STEP 5: CREATING RISK LABELS")
print("="*80)

from src.risk import RiskLabeler, RiskForecaster

# Pass config_path explicitly
risk_labeler = RiskLabeler(config_path=config_path)
train_features_risk = risk_labeler.fit_transform(train_features, target_col='forward_returns')
print(f"✓ Risk labels created: {train_features_risk['risk_label'].notna().sum()} valid samples")

# ==================== STEP 6: Feature Selection for Risk Model ====================
print("\n" + "="*80)
print("STEP 6: FEATURE SELECTION FOR RISK MODEL")
print("="*80)

train_risk_selected, risk_features = fe.select_features_by_importance(
    train_features_risk,
    target_col='risk_label',
    method='correlation',
    top_n=200
)

train_risk_selected, _ = fe.remove_correlated_features(
    train_risk_selected,
    threshold=0.95,
    target_col='risk_label'
)

RISK_FEATURE_COLS = [
    col for col in train_risk_selected.columns 
    if col not in ['date_id', 'forward_returns', 'risk_free_rate', 
                   'market_forward_excess_returns', 'risk_label']
]
print(f"✓ Risk feature selection complete: {len(RISK_FEATURE_COLS)} features")

# ==================== STEP 7: Train Risk Model ====================
print("\n" + "="*80)
print("STEP 7: TRAINING RISK PREDICTION MODEL")
print("="*80)

risk_forecaster = RiskForecaster(config_path=config_path)

# Use train() method instead of train_cv()
oof_risk, risk_models = risk_forecaster.train(
    df=train_risk_selected,
    feature_cols=RISK_FEATURE_COLS,
    risk_col='risk_label',
    n_folds=5,
    early_stopping_rounds=50
)

# Calculate OOF score
valid_idx = ~np.isnan(oof_risk)
oof_rmse = np.sqrt(np.mean((train_risk_selected.loc[valid_idx, 'risk_label'].values - oof_risk[valid_idx])**2))

print(f"\n✅ Risk model trained!")
print(f"OOF Score (RMSE): {oof_rmse:.6f}")

print("\n" + "="*80)
print("✅ FULL PIPELINE COMPLETE - READY FOR INFERENCE")
print("="*80)
print(f"Return model features: {len(FEATURE_COLS)}")
print(f"Risk model features: {len(RISK_FEATURE_COLS)}")


## 4️⃣ Define Prediction Function

This function will be called for each timestep.
It receives a Polars DataFrame and must return a single float value.

In [ ]:
def predict(test: pl.DataFrame) -> float:
    """
    Predict allocation using Return and Risk models with Quantile Binning strategy.
    
    Args:
        test: Polars DataFrame with test data for current timestep
        
    Returns:
        float: Optimal allocation (0.0 to 2.0)
    """
    # Convert Polars to Pandas
    test_df = test.to_pandas()
    
    # Feature engineering
    test_features = fe.transform(test_df)
    
    # ===== Return Prediction =====
    X_return = test_features[FEATURE_COLS]
    r_hat = predictor.predict(X_return)[0]
    
    # ===== Risk Prediction =====
    X_risk = test_features[RISK_FEATURE_COLS]
    sigma_hat = risk_forecaster.predict(X_risk)[0]
    
    # ===== Quantile Binning Strategy =====
    # Optimized locally: bins=7, allocations=[0.0, 0.0, 0.09, 0.74, 0.90, 1.08, 1.85]
    # Score: 0.124 (Sharpe ratio with vol constraint)
    
    # Calculate risk-adjusted return (Sharpe-like ratio)
    if sigma_hat > 0:
        risk_adjusted_return = r_hat / sigma_hat
    else:
        risk_adjusted_return = 0.0
    
    # Map to quantile bins (based on risk-adjusted return)
    if risk_adjusted_return < -1.0:
        allocation = 0.0  # Very negative → no investment
    elif risk_adjusted_return < -0.5:
        allocation = 0.0  # Negative → no investment
    elif risk_adjusted_return < 0.0:
        allocation = 0.09  # Slightly negative → minimal
    elif risk_adjusted_return < 0.5:
        allocation = 0.74  # Slightly positive → below market
    elif risk_adjusted_return < 1.0:
        allocation = 0.90  # Positive → near market
    elif risk_adjusted_return < 1.5:
        allocation = 1.08  # Strong positive → above market
    else:
        allocation = 1.85  # Very strong → high leverage
    
    # Safety clip to valid range
    allocation = np.clip(allocation, 0.0, 2.0)
    
    return float(allocation)

print("✅ predict() function defined")
print("\nFunction signature: predict(test: pl.DataFrame) -> float")
print("  - Input: Polars DataFrame with test data")
print("  - Output: Optimal allocation (0.0 to 2.0)")
print(f"  - Return model features: {len(FEATURE_COLS)}")
print(f"  - Risk model features: {len(RISK_FEATURE_COLS)}")
print("\nStrategy: Quantile Binning with Risk-Adjusted Returns")
print("  Bins: 7 quantiles based on r_hat / sigma_hat")
print("  Allocations: [0.0, 0.0, 0.09, 0.74, 0.90, 1.08, 1.85]")
print("  Optimized Score: 0.124 (local CV)")


## 5️⃣ Initialize Inference Server

In [ ]:
print("="*80)
print("INITIALIZING INFERENCE SERVER")
print("="*80)

if not KAGGLE_ENV:
    print("\n⚠️  Skipping server initialization (not in Kaggle environment)")
    print("Please run this notebook on Kaggle platform")
else:
    # Create inference server with our predict function
    inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)
    
    print("\n✅ Inference server initialized!")
    print("\nServer configuration:")
    print("  - Prediction function: predict()")
    print("  - Response time limit: 5 minutes per prediction")
    print("  - Server startup limit: 15 minutes")
    print("\n⚡ Ready to serve predictions!")

## 6️⃣ Start Server (Kaggle Evaluation Mode)

**This cell starts the actual server when running on Kaggle's evaluation system.**

- In competition mode: Waits for timestep-by-timestep data from Kaggle
- In local mode: Runs local gateway for testing

In [ ]:
print("="*80)
print("GENERATING PREDICTIONS FOR TEST DATA")
print("="*80)

# Load test data
test_path = "/kaggle/input/hull-tactical-market-prediction/test.csv"
test_df = pd.read_csv(test_path)
print(f"✓ Test data loaded: {test_df.shape}")

# Generate predictions for all test samples
print("\nGenerating predictions...")
predictions = []

for idx in range(len(test_df)):
    # Get single row
    test_row = test_df.iloc[[idx]]
    
    # Apply feature engineering
    test_features = fe.transform(test_row)
    
    # Handle missing features (fill with 0)
    for col in FEATURE_COLS:
        if col not in test_features.columns:
            test_features[col] = 0.0
    
    for col in RISK_FEATURE_COLS:
        if col not in test_features.columns:
            test_features[col] = 0.0
    
    # Select features
    X_return = test_features[FEATURE_COLS]
    X_risk = test_features[RISK_FEATURE_COLS]
    
    # Predict
    r_hat = predictor.predict(X_return)[0]
    sigma_hat = risk_forecaster.predict(X_risk)[0]
    
    # Calculate risk-adjusted return
    if sigma_hat > 0:
        risk_adjusted_return = r_hat / sigma_hat
    else:
        risk_adjusted_return = 0.0
    
    # Quantile binning
    if risk_adjusted_return < -1.0:
        allocation = 0.0
    elif risk_adjusted_return < -0.5:
        allocation = 0.0
    elif risk_adjusted_return < 0.0:
        allocation = 0.09
    elif risk_adjusted_return < 0.5:
        allocation = 0.74
    elif risk_adjusted_return < 1.0:
        allocation = 0.90
    elif risk_adjusted_return < 1.5:
        allocation = 1.08
    else:
        allocation = 1.85
    
    allocation = np.clip(allocation, 0.0, 2.0)
    predictions.append(allocation)
    
    if (idx + 1) % 100 == 0:
        print(f"  Processed {idx + 1}/{len(test_df)} samples...")

predictions = np.array(predictions)
print(f"\n✓ Predictions generated: {len(predictions)} samples")

# Create submission DataFrame
submission = pd.DataFrame({
    'date_id': test_df['date_id'].astype('int64'),
    'allocation': predictions.astype('float64')
})

# Clip to valid range
submission['allocation'] = submission['allocation'].clip(0, 2)

# Validation
print("\n" + "="*80)
print("VALIDATING SUBMISSION")
print("="*80)

assert list(submission.columns) == ['date_id', 'allocation'], "❌ Wrong column names!"
assert submission['allocation'].isna().sum() == 0, "❌ Contains NaN values!"
assert (submission['allocation'] >= 0).all(), "❌ Contains values < 0!"
assert (submission['allocation'] <= 2).all(), "❌ Contains values > 2!"

print("✓ Column names: ['date_id', 'allocation']")
print(f"✓ No missing values: {submission['allocation'].isna().sum()} NaN")
print(f"✓ Valid range: [{submission['allocation'].min():.4f}, {submission['allocation'].max():.4f}]")
print(f"✓ Mean allocation: {submission['allocation'].mean():.4f}")

# Save as Parquet
output_path = '/kaggle/working/submission.parquet'
submission.to_parquet(
    output_path,
    index=False,
    engine='pyarrow'
)
# Preview submission
print(submission.head(10))
print(f"\nAllocation distribution:")
print(submission['allocation'].value_counts().sort_index())
print(f"\nStats:")
print(submission['allocation'].describe())

print(f"\n✅ Submission saved to: {output_path}")
print(f"   Size: {len(submission)} rows")
print("\n" + "="*80)
print("🎉 READY FOR SUBMISSION!")
print("="*80)


## 📊 Pipeline Summary

### Training Pipeline (Runs Once on Startup):

```
Raw Data (train.csv)
    ↓
[1] Preprocessing (winsorize, normalize, scale)
    ↓
[2] Feature Engineering (rolling, lag, interaction, technical)
    ↓  
[3] Feature Selection (correlation-based, top 150)
    ↓
[4] Model Training (LightGBM with CV)
    ↓
Ready for Inference! 🎯
```

### Inference Pipeline (Runs Per Timestep):

```
New Data (test row)
    ↓
[1] Same Preprocessing
    ↓
[2] Same Feature Engineering  
    ↓
[3] Same Feature Selection
    ↓
[4] Predict Return (r_hat)
    ↓
[5] Convert to Allocation (0 to 2)
    ↓
Return Single Float ✅
```

### Key Components:

| Component | What It Does |
|-----------|--------------|
| **DataLoader** | Loads, cleans, preprocesses data |
| **FeatureEngineering** | Creates 600+ features, selects best 150 |
| **ReturnPredictor** | LightGBM model for return prediction |
| **predict()** | Main inference function (return → allocation) |

### Model Architecture:

- **Algorithm**: LightGBM (gradient boosting)
- **Features**: ~150 selected features
- **CV Strategy**: 5-fold time series walk-forward
- **Target**: Forward returns
- **Output**: Allocation (0.0 to 2.0)

### Strategy:

**Simple Return-Based Allocation:**
```python
allocation = clip(1.0 + 50 * predicted_return, 0.0, 2.0)
```

- Positive return prediction → increase allocation (up to 2.0)
- Negative return prediction → decrease allocation (down to 0.0)
- Neutral prediction (0) → baseline allocation (1.0)

### Performance Expectations:

Based on local optimization:
- **OOF Score**: ~0.12 - 0.15 (expected on CV)
- **Volatility Ratio**: ~1.1 - 1.2 (within constraint)
- **Leverage**: Minimal (mostly < 10%)

### 🎯 Ready for Submission!

1. ✅ ZIP created: `prediction_market_modules.zip`
2. ✅ Upload to Kaggle Datasets
3. ✅ Add dataset to this notebook
4. ✅ Click "Run All"
5. ✅ Submit to competition!

**Good luck! 🚀**